# 01 · Pipeline de datos (T-04, bloque A)

**Dónde:** `pipeline/src/data/limpieza.py`. **Por qué:** los datos crudos traen duplicados, lecturas nulas,
retrocesos de medidor e importes con una tarifa inventada; además hay que definir el objetivo `pago_tardio`
sin fuga de información. **Cómo:** carga tipada, deduplicación, marcas de calidad (no se imputa),
recálculo de importes con el tarifario CEA real (T-03), censura por `FECHA_CORTE` y `merge` de recibos,
lecturas y tomas. **Con qué datos:** los cinco CSV de `data/simulados/` (simulación declarada, sin datos personales).

In [1]:
import sys, pathlib
raiz = pathlib.Path.cwd()
while not (raiz / "pipeline").exists() and raiz != raiz.parent:
    raiz = raiz.parent
sys.path.insert(0, str(raiz))
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.max_colwidth", 120)
from pipeline.notebooks.graficas import dibujar, tabla

In [2]:
from pipeline.src.data.limpieza import ejecutar
tablas, informe = ejecutar()
informe

,Paso,Resultado
0,Carga de tomas,"480 filas x 10 columnas, 0 celdas nulas"
1,Carga de lecturas,"14,486 filas x 8 columnas, 230 celdas nulas"
2,Carga de recibos,"14,400 filas x 14 columnas, 1989 celdas nulas"
3,Carga de telemetria,"115,200 filas x 4 columnas, 0 celdas nulas"
4,Carga de quejas,"1,400 filas x 7 columnas, 0 celdas nulas"
5,"Lecturas: duplicados por (toma, periodo)",86 filas eliminadas (se conserva la lectura más reciente)
6,Lecturas: nulas,114 marcadas (no se imputan)
7,Lecturas: retrocesos de medidor,57 marcados como inconsistentes
8,Recibos: importes con tarifario CEA real,"14400 recibos recalculados; total facturado 5,997,544 -> 19,405,612 MXN (mapeo {'domestico': 'domestico_medio', 'com..."
9,Recibos: pagados sin fecha de pago,67 recibos; se conserva la etiqueta original (la fecha no se imputa)


In [3]:
for nombre, df in tablas.items():
    print(f"{nombre:11s} {df.shape}")
tablas["dataset"].dtypes.value_counts()

tomas       (480, 8)
lecturas    (14400, 11)
recibos     (14400, 17)
telemetria  (115200, 4)
quejas      (1400, 7)
dataset     (14400, 33)


float64           10
bool               8
datetime64[us]     6
str                5
int32              2
boolean            1
int64              1
Name: count, dtype: int64

In [4]:
tablas["dataset"].head()

,id_toma,periodo,fecha_emision,fecha_vencimiento,fecha_pago,pagado,consumo_m3,importe_agua,importe_alcantarillado,importe_saneamiento,...,privada,tipo_ocupante,incluye_alcantarillado,incluye_saneamiento,fecha_alta,periodo_fecha,anio,mes,antiguedad_meses,dias_atraso
0,T00048F12E2,2024-04,2024-04-24,2024-05-14,2024-05-13,True,12.7,521.89,52.19,62.63,...,Santa Teresa,domestico,True,True,2023-02-20,2024-04-01,2024,4,13.3,0.0
1,T00048F12E2,2024-05,2024-05-25,2024-06-14,2024-06-02,True,11.1,466.28,46.63,55.95,...,Santa Teresa,domestico,True,True,2023-02-20,2024-05-01,2024,5,14.3,0.0
2,T00048F12E2,2024-06,2024-06-30,2024-07-20,2024-07-09,True,0.0,225.82,22.58,27.10,...,Santa Teresa,domestico,True,True,2023-02-20,2024-06-01,2024,6,15.3,0.0
3,T00048F12E2,2024-07,2024-07-27,2024-08-16,2024-08-06,True,9.3,417.16,41.72,50.06,...,Santa Teresa,domestico,True,True,2023-02-20,2024-07-01,2024,7,16.3,0.0
4,T00048F12E2,2024-08,2024-08-26,2024-09-15,2024-08-30,True,8.6,417.16,41.72,50.06,...,Santa Teresa,domestico,True,True,2023-02-20,2024-08-01,2024,8,17.3,0.0


## Conclusiones

In [5]:
for fila in informe.itertuples():
    print(f"- {fila.Paso}: {fila.Resultado}")

- Carga de tomas: 480 filas x 10 columnas, 0 celdas nulas
- Carga de lecturas: 14,486 filas x 8 columnas, 230 celdas nulas
- Carga de recibos: 14,400 filas x 14 columnas, 1989 celdas nulas
- Carga de telemetria: 115,200 filas x 4 columnas, 0 celdas nulas
- Carga de quejas: 1,400 filas x 7 columnas, 0 celdas nulas
- Lecturas: duplicados por (toma, periodo): 86 filas eliminadas (se conserva la lectura más reciente)
- Lecturas: nulas: 114 marcadas (no se imputan)
- Lecturas: retrocesos de medidor: 57 marcados como inconsistentes
- Recibos: importes con tarifario CEA real: 14400 recibos recalculados; total facturado 5,997,544 -> 19,405,612 MXN (mapeo {'domestico': 'domestico_medio', 'comercial': 'comercial', 'publico': 'publico_oficial', 'industrial': 'industrial'})
- Recibos: pagados sin fecha de pago: 67 recibos; se conserva la etiqueta original (la fecha no se imputa)
- Recibos: vencidos y sin pagar a la fecha de corte: 122 etiquetados como pago tardío
- Recibos: censura por fecha de co